# 05D Real COF machine-learning datasets

This chapter removes synthetic targets from the main exercise and works directly with published COF datasets: **real public features + molecular-simulation targets + provenance + baseline models**.

We use three complementary sources: **COFSpace / CoRE COFs**, **CURATED-COFs adsorption data**, and the **ReDD-COFFEE CO₂-capture HTS dataset**. Here, ‘real target’ means a value released by the original research dataset; it does not imply an experimental measurement.

## 1. Dataset A — COFSpace: CO₂ adsorption for 1060 CoRE COFs

COFSpace publishes ML-ready feature tables and simulated gas-uptake data. The 1-bar CO₂ table contains PLD, LCD, accessible surface area, porosity, Henry coefficient, elemental fractions and CO₂ uptake.

In [ ]:
import pandas as pd
import numpy as np
url = 'https://raw.githubusercontent.com/gokhanonderaksu/COFSpace/main/OnlyCoRECOF%20-%20Feature%20Sets/CoRECOF%20-%20CO2%20-%201%20BAR.csv'
cofspace = pd.read_csv(url)
print(cofspace.shape)
display(cofspace.head())

### A first real adsorption baseline

This is deliberately a baseline, not an attempt to reproduce the paper's optimized model. `KCO2` is an affinity/energy-related feature: retain it when reproducing that surrogate setting, or remove it when testing prediction from cheaper structural/chemical descriptors only.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, r2_score
target = 'CO2-1 bar (mol/kg)'
features = [c for c in cofspace.columns if c != target]
X, y = cofspace[features], cofspace[target]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
model = RandomForestRegressor(n_estimators=300, random_state=42, n_jobs=-1).fit(X_train, y_train)
pred = model.predict(X_test)
print('MAE:', mean_absolute_error(y_test, pred))
print('R2 :', r2_score(y_test, pred))
display(pd.Series(model.feature_importances_, index=features).sort_values(ascending=False).to_frame('importance'))

## 2. Dataset B — CURATED-COFs adsorption properties

`nachatz/cof-data` organizes adsorption tasks derived from CURATED-COFs / Materials Cloud. `properties.csv` contains H₂, O₂, CO₂, CH₄, N₂, Xe, Kr, H₂O and H₂S properties, while `simple_features.csv` contains ASA, density and a pore-size descriptor. The stable COF identifier connects the tables. This is a useful exercise in **joining structure/features and properties by material ID**.

In [ ]:
prop_url = 'https://raw.githubusercontent.com/nachatz/cof-data/main/properties.csv'
feat_url = 'https://raw.githubusercontent.com/nachatz/cof-data/main/simple_features.csv'
properties = pd.read_csv(prop_url)
simple_features = pd.read_csv(feat_url)
dataset_b = simple_features.merge(properties, left_on='cof', right_on='name', how='inner')
print('features:', simple_features.shape, 'properties:', properties.shape, 'merged:', dataset_b.shape)
display(dataset_b[['cof','ASA_m^2/g','Density','LS','co2_30bar','co2_ads_unit','co2_henry','co2_henry_unit']].head())

In [ ]:
df = dataset_b[['ASA_m^2/g','Density','LS','co2_30bar']].dropna()
X, y = df[['ASA_m^2/g','Density','LS']], df['co2_30bar']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
m2 = RandomForestRegressor(n_estimators=300, random_state=42, n_jobs=-1).fit(X_train, y_train)
p2 = m2.predict(X_test)
print('n =', len(df), 'MAE =', mean_absolute_error(y_test, p2), 'R2 =', r2_score(y_test, p2))

## 3. Dataset C — ReDD-COFFEE CO₂-capture high-throughput screening

`SupportingInformation_CO2captureHTS_2024` is a research-scale workflow with `features.csv`, `results.csv`, fixed `structs_train.txt` / `structs_test.txt`, feature reduction, prediction and SHAP analysis. The full feature archive is large, so the course inspects the results table directly and treats the complete archive as an advanced project rather than downloading it automatically in every Colab runtime.

Repository: https://github.com/jsdvos/SupportingInformation_CO2captureHTS_2024

In [ ]:
results_url = 'https://raw.githubusercontent.com/jsdvos/SupportingInformation_CO2captureHTS_2024/master/Step2_MachineLearning/data/input/results.csv'
redd_results = pd.read_csv(results_url, sep=';')
print(redd_results.shape)
display(redd_results.head())
print(list(redd_results.columns))

## 4. What does each dataset teach?

| Dataset | Type | Best teaching use |
|---|---|---|
| COFSpace / CoRE COF | ~10³, simulated labels | supervised regression, feature importance, gas/pressure-dependent targets |
| CURATED-COFs adsorption | experimental COF structures + computed properties | CIF/ID/property joins, units and conditions, small-data baselines |
| ReDD-COFFEE HTS | ~10⁵ hypothetical COFs | fixed splits, feature reduction, surrogate screening, SHAP and scale |

Do not concatenate these datasets blindly. Their structure sources, simulation protocols, descriptor definitions and target conditions differ. Treat cross-dataset use as transfer learning, external validation or domain-shift analysis only after checking compatibility.

## 5. Exercises

1. Compare CO₂ models at 0.1, 1, 5 and 10 bar in COFSpace.
2. Predict both `co2_30bar` and `co2_henry` in Dataset B and compare controlling features.
3. Merge composition descriptors calculated from CIFs in 05B with Dataset B through COF IDs.
4. For ReDD-COFFEE, use the authors' fixed train/test lists rather than generating a new random split.
5. Write a dataset card for every experiment: source, structure type, target, T/P, units, simulation method, features, split and citation/license.

## Sources and citation

- COFSpace: G. Onder Aksu et al., *The COF Space: Materials Features, Gas Adsorption, and Separation Performances Assessed by Machine Learning*. Data/scripts: https://github.com/gokhanonderaksu/COFSpace
- CURATED-COFs / Materials Cloud: D. Ongari et al., *Building a consistent and reproducible database for adsorption evaluation in Covalent-Organic Frameworks*. Data DOI: 10.24435/materialscloud:z6-jn.
- ReDD-COFFEE CO₂-capture HTS: https://github.com/jsdvos/SupportingInformation_CO2captureHTS_2024

For research use, cite the original papers and dataset records rather than only this tutorial.